In [1]:
# nSTAT-python notebook example: NetworkTutorial
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
SRC_PATH = (REPO_ROOT / "src").resolve()
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from nstat import Analysis, Covariate, FitResSummary, History, Trial, TrialConfig
from nstat.ConfigColl import ConfigColl
from nstat.CovColl import CovColl
from nstat.notebook_figures import FigureTracker
from nstat.simulators import simulate_two_neuron_network

np.random.seed(0)
OUTPUT_ROOT = REPO_ROOT / "output" / "notebook_images"
__tracker = FigureTracker(topic='NetworkTutorial', output_root=OUTPUT_ROOT, expected_count=4)

# SECTION 0: Section 0
# Author: Iahn Cajigas
# Date: 2/10/2014


In [2]:
# SECTION 1: Point Process Network Simulation
# In order to understand how the point process GLM framework can be used to estimate the network connectivity within a population of neurons, we simulate a network of 2 neurons.
# This block diagram specifies a conditional intensity function of the form
# lambda_{i} \cdot \Delta = logistic(\mu_{i} + H*\Delta N_{i}[n] +
# S*u_{stim}[n] + E*\Delta N_{k}[n]
# where, \hbox{\fontsize{14}{16}\selectfont\(logistic(x)=e^{x}/{1+e^{x}}\)}. Note that * is the convolution opertator.

In [3]:
# SECTION 2: 2 Neuron Network
plt.close("all")
Ts = 0.001
sampleRate = 1.0 / Ts
numNeurons = 2
tMax = 50.0
windowTimes = [0.0, Ts, 2 * Ts, 3 * Ts]
network = simulate_two_neuron_network(duration_s=tMax, dt=Ts, seed=4)
time = network.time
baseline_mu = network.baseline_mu
history_kernel = network.history_kernel
stim_kernel = network.stimulus_kernel
ensemble_kernel = network.ensemble_kernel
actual_network = network.actual_network


In [4]:
# SECTION 3: Baseline firing rate of the neurons being modeled

In [5]:
# SECTION 4: History Effect
# Captures how the firing of a neuron at modulates its probability of firing. Captures effects such as the refractory period and bursting. We use the same firing history for both neurons in this example. Note that the firing activity at time n leads to strong inhibition at time n+1 (refractory period) and that this effect becomes smaller over the next two time periods.
# 1*h[n]=-4*\Delta N[n-1]-2*\Delta N[n-2] -1*\Delta N[n-3]
# Note that the one sample delay in same cell firing is included in the simulink model.

In [6]:
# SECTION 5: Stimulus Effect
# 1*s_{1}[n]=1*u_{stim}[n]
# 1*s_{2}[n]=-1*u_{stim}[n]
# Neuron 1 is positively modulated by the stimulus
# Neuron 1 is negatively modulated by the stimulus

In [7]:
# SECTION 6: Ensemble Effect
# Captures the effect of how neighboring neuron firing modulates the firing of a given neuron.
# 1*e_{1}[n]=1*\Delta N_{2}[n-1]
# 1*e_{2}[n]=-4*\Delta N_{1}[n-1]
# Note that the one sample delay in firing of the neighbor cell is included in the simulink model.
# Neuron 2 firing positively modulates Neuron 1
# Neuron 1 firing has strong inhibitory effect on neuron 2.

In [8]:
# SECTION 7: Stimulus
# We use a simple sine wave here but we may want to explore other types of inputs to see if they affect the recovery of the network parameters.
f = 1
stimCov = Covariate(time, network.latent_drive, "Stimulus", "time", "s", "Voltage", ["sin"])
baselineCov = Covariate(time, np.ones_like(time), "Baseline", "time", "s", "", ["mu"])


In [9]:
# SECTION 8: Simulate the Network
# Uses a binomial model for the conditional intensity function; this reproduces the MATLAB tutorial workflow with a native Python simulation.
fitType = "binomial"
Algorithm = "BNLRCG" if fitType == "binomial" else "GLM"
spikeColl = network.spikes
trial = Trial(spikeColl, CovColl([stimCov, baselineCov]), None, History(windowTimes))
trial.setEnsCovHist(windowTimes)

fig = __tracker.new_figure("network-simulation-overview")
fig.clear()
ax1, ax2 = fig.subplots(2, 1, sharex=True)
ax1.plot(time, network.lambda_delta[:, 0], label="Neuron 1")
ax1.plot(time, network.lambda_delta[:, 1], label="Neuron 2")
ax1.set_ylabel("spike probability")
ax1.set_title("Simulated conditional intensity (binomial)")
ax1.legend(loc="upper right")
stimCov.plot(handle=ax2)
ax2.set_title("Stimulus drive")
ax2.set_xlim(0.0, tMax / 10.0)
fig.tight_layout()

fig = __tracker.new_figure("network-raster")
fig.clear()
ax = fig.subplots(1, 1)
spikeColl.plot(handle=ax)
ax.set_xlim(0.0, tMax / 10.0)
ax.set_title("Simulated 2-neuron raster")
fig.tight_layout()

print({
    "fitType": fitType,
    "algorithm": Algorithm,
    "num_neurons": spikeColl.numSpikeTrains,
    "num_spikes": [spikeColl.getNST(i + 1).n_spikes for i in range(numNeurons)],
})


{'fitType': 'binomial', 'algorithm': 'BNLRCG', 'num_neurons': 2, 'num_spikes': [2590, 2365]}


In [10]:
# SECTION 9: GLM Model Fitting Setup
# In this section, we create the appropriate structures to fit several GLM models to the data generated above.
# Create a constant covariate representing the mean firing rate $$\mu_{i}$
#
# Use stimulation and baseline as possible covariates
# trial.setTrialPartition([0 tMax/2 tMax]);

In [11]:
# SECTION 10: GLM Model Fitting and Results
configs = ConfigColl([
    TrialConfig([["Baseline", "mu"]], sampleRate, [], [], [], name="Baseline"),
    TrialConfig([["Baseline", "mu"]], sampleRate, [], [0.0, Ts], [], name="Baseline+EnsHist"),
    TrialConfig([["Baseline", "mu"], ["Stimulus", "sin"]], sampleRate, windowTimes, [0.0, Ts], [], name="Stim+Hist+EnsHist"),
])
results = Analysis.RunAnalysisForAllNeurons(trial, configs, 0, Algorithm)
if not isinstance(results, list):
    results = [results]
summary = FitResSummary(results)

fig = __tracker.new_figure("network-summary")
summary.plotSummary(handle=fig)

est_network = np.zeros((2, 2), dtype=float)
for neuron_idx, fit in enumerate(results, start=1):
    coeffs, labels, _ = fit.getCoeffsWithLabels(3)
    for coeff, label in zip(coeffs, labels, strict=False):
        if neuron_idx == 1 and str(label).startswith("2:"):
            est_network[0, 1] = coeff
        if neuron_idx == 2 and str(label).startswith("1:"):
            est_network[1, 0] = coeff

fig = __tracker.new_figure("network-actual-vs-estimated")
fig.clear()
ax1, ax2 = fig.subplots(1, 2)
im1 = ax1.imshow(actual_network, cmap="coolwarm", vmin=-4.0, vmax=1.0)
ax1.set_title("Actual")
ax1.set_xticks([0, 1], ["N1", "N2"])
ax1.set_yticks([0, 1], ["N1", "N2"])
ax2.imshow(est_network, cmap="coolwarm", vmin=-4.0, vmax=1.0)
ax2.set_title("Estimated 1ms")
ax2.set_xticks([0, 1], ["N1", "N2"])
ax2.set_yticks([0, 1], ["N1", "N2"])
fig.colorbar(im1, ax=[ax1, ax2], shrink=0.8)
fig.tight_layout()
print({"config_names": summary.fitNames, "estimated_network": est_network.tolist()})
__tracker.finalize()


{'config_names': ['Baseline', 'Baseline+EnsHist', 'Stim+Hist+EnsHist'], 'estimated_network': [[0.0, 0.0], [0.0, 0.0]]}


/var/folders/tg/z6dfb8b13wg_h4f3v8whzpgh0000gn/T/ipykernel_95209/2639979657.py:36: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/Users/iahncajigas/Library/CloudStorage/Dropbox/Codex/nSTAT-python/nstat/notebook_figures.py:42: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  self._active_fig.tight_layout()
